# BRFSS 2025 — Initial Data Exploration

Purpose: verify the official CDC archive, load the SAS Transport dataset, inspect shape/columns, and examine the raw `DIABETE4` target before any cleaning or modeling.

**No model training and no target recoding are performed in this notebook.**


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import hashlib
import json
import pandas as pd
from IPython.display import display

TARGET = "DIABETE4"
EXPECTED_XPT_BYTES = 799_971_280

xpt_candidates = list(Path("/kaggle/input").rglob("LLCP2025.XPT"))
zip_candidates = list(Path("/kaggle/input").rglob("LLCP2025XPT.zip"))

print("XPT candidates:", [str(p) for p in xpt_candidates])
print("ZIP candidates:", [str(p) for p in zip_candidates])

if len(xpt_candidates) == 1:
    xpt_path = xpt_candidates[0]
    source_mode = "Kaggle dataset extracted XPT"
elif len(xpt_candidates) == 0 and len(zip_candidates) == 1:
    zip_path = zip_candidates[0]
    working_dir = Path("/kaggle/working/brfss_2025")
    working_dir.mkdir(parents=True, exist_ok=True)

    with ZipFile(zip_path) as zf:
        members = [info for info in zf.infolist() if not info.is_dir()]
        if len(members) != 1:
            raise RuntimeError(f"Expected one data payload, found {len(members)}.")
        member = members[0]
        normalized_name = Path(member.filename.strip()).name
        if normalized_name != "LLCP2025.XPT":
            raise RuntimeError(f"Unexpected payload name: {normalized_name!r}")
        xpt_path = working_dir / normalized_name
        with zf.open(member) as src, xpt_path.open("wb") as dst:
            for chunk in iter(lambda: src.read(8 * 1024 * 1024), b""):
                dst.write(chunk)
    source_mode = "Kaggle dataset ZIP fallback"
else:
    raise RuntimeError(
        "Expected exactly one BRFSS input payload from Kaggle Dataset. "
        f"Found {len(xpt_candidates)} XPT and {len(zip_candidates)} ZIP candidates."
    )

actual_bytes = xpt_path.stat().st_size
print("Source mode:", source_mode)
print("XPT path:", xpt_path)
print("XPT bytes:", f"{actual_bytes:,}")

if actual_bytes != EXPECTED_XPT_BYTES:
    raise RuntimeError(
        f"Unexpected XPT size: {actual_bytes:,}; expected {EXPECTED_XPT_BYTES:,}."
    )

sha256 = hashlib.sha256()
with xpt_path.open("rb") as f:
    for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
        sha256.update(chunk)
xpt_sha256 = sha256.hexdigest()
print("XPT SHA-256:", xpt_sha256)


In [ ]:
print("Validated Kaggle input payload.")
print("Ready to load SAS Transport file:", xpt_path)


In [ ]:
df = pd.read_sas(xpt_path, format="xport", encoding="utf-8")

print("Shape:", df.shape)
print("Rows:", f"{len(df):,}")
print("Columns:", df.shape[1])
print("Memory usage:", f"{df.memory_usage(deep=True).sum() / (1024**3):.2f} GiB")

if TARGET not in df.columns:
    raise RuntimeError(f"Target {TARGET!r} not found in dataset.")

display(df.head())


In [ ]:
target_counts = (
    df[TARGET]
    .value_counts(dropna=False)
    .rename_axis(TARGET)
    .reset_index(name="count")
)
target_counts["percent"] = target_counts["count"] / len(df) * 100

print("Raw DIABETE4 distribution (no recoding yet):")
display(target_counts)
target_counts.to_csv("/kaggle/working/target_distribution.csv", index=False)


In [ ]:
column_summary = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(dtype) for dtype in df.dtypes],
    "missing_count": df.isna().sum().values,
    "missing_pct": (df.isna().mean().values * 100),
    "n_unique": [df[col].nunique(dropna=True) for col in df.columns],
}).sort_values(["missing_pct", "column"], ascending=[False, True])

print("Columns with the highest missingness:")
display(column_summary.head(30))

print("All column names:")
print(df.columns.tolist())

column_summary.to_csv("/kaggle/working/column_summary.csv", index=False)


In [ ]:
overview = {
    "source_mode": source_mode,
    "source_file": xpt_path.name,
    "xpt_sha256": xpt_sha256,
    "xpt_bytes": int(actual_bytes),
    "rows": int(df.shape[0]),
    "columns": int(df.shape[1]),
    "target": TARGET,
    "target_missing": int(df[TARGET].isna().sum()),
    "memory_gib": round(float(df.memory_usage(deep=True).sum() / (1024**3)), 3),
}

with open("/kaggle/working/dataset_overview.json", "w", encoding="utf-8") as f:
    json.dump(overview, f, indent=2)

print(json.dumps(overview, indent=2))
print("Initial exploration complete.")
